In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet("data/pq/green/*/*")

In questo notebook si prende la seguente query: 

    SELECT
        date_trunc('hour', lpep_pickup_datetime) AS hour,
        PULocationID AS zone,
        SUM(total_amount) AS amount, 
        COUNT(1) AS number_records
    FROM green
    WHERE  lpep_pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY 1,2
    ORDER BY 1,2

Quello che si fa è realizzare questa query tramite gli RDD 
(ovviamente è più macchinoso rispetto a quello che abbiamo visto con i Dataframe nel notebook precedente). 

In [ ]:
#questo comando corrisponde solo alla select dei campi (quelli senza trasformazioni (sum, truncate ecc))
rdd = df_green.select("lpep_pickup_datetime", "PULocationID", "total_amount").rdd

In [ ]:
from datetime import datetime

In [ ]:
#adesso si definisce il filtro della where 
start = datetime(year = 2020, month = 1, day = 1)

#si definiscce una funzione Python, ma si potrebbe fare anche 
# direttamente con una lambda function, che prende in input una riga del dataframe e 
# restituisce un booleano (True se la riga deve essere mantenuta, 
# False se deve essere scartata).


def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [ ]:
#nota: si chiama la funzione filter_outliers definita sopra. 
#al posto della nome della funzione si potrebbe scrivere: 
#rdd.filter(lambda row: row.lpep_pickup_datetime >= start).take(5)

rdd.filter(filter_outliers).take(5)

NOTA: nell'ultimo output ci sono delle Row che fanno parte degli RDD 

In [ ]:
#dalla struttura di sopra, si prende una singola riga e sotto poi si accede al campo lpep_pickup_datetime.
rows = rdd.take(10)
row = rows[0]

In [ ]:
row.lpep_pickup_datetime

In [ ]:
#definizione della funzione che fa la map per <K,V> 
#in cui la Key = hour e zone 
#Value = amount e value

def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1 
    value = (amount, count)

    return (key, value)

In [ ]:
rdd.filter(filter_outliers).map(prepare_for_grouping).take(5)

Si definisce una funzione di Reduce by Key

Dalla query di partenza si realizza la parte: 

    SUM(total_amount) AS amount, 
    COUNT(1) AS number_records

In [ ]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)


In [ ]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .take(5)

NOTA: l'output appena ottenuto non è facilmente leggibile. 
Quindi per renderlo più leggibile a livello umano, si trasforma il risultato da RDD a Dataframe e si rende un-nested cosi si legge meglio.

In [ ]:
#si definisce la funzione che fa l'un-packing dei valori della riga (Row)
def unwrap(row): 
    return (row[0][0], row[0][1], row[1][0], row[1][1])

In [ ]:
#qui si trasforma in Dataframe tramite la funzione .toDF()
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF() \
    .show()

NOTA: nell'ultimo output si può vedere come manca lo schema della tabella ma ci sono dei numeri. 

Per inserire lo schema bisogna fare: 

In [ ]:
from collections import namedtuple

In [ ]:
RevenueRow = namedtuple("RevenueRow", ["hour", "zone", "amount", "count"])

In [ ]:
def unwrap(row): 
    return RevenueRow(
        hour = row[0][0], 
        zone = row[0][1], 
        amount = row[1][0], 
        count = row[1][1])

In [ ]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF() \
    .show(5)

In [ ]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF()

In [ ]:
from pyspark.sql import types

In [ ]:
#si impone lo schema al dataframe: 
result_schema = types.StructType([
  types.StructField('hour', types.TimestampType(), True), 
  types.StructField('zone', types.IntegerType(), True), 
  types.StructField('amount', types.DoubleType(), True), 
  types.StructField('count',types.IntegerType(), True)
])

In [ ]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [ ]:
df_result.write.parquet("tmp/green-revenue")

Parte su RDD MapPartition

il **MapPartiton** è simile all'operazione della Map, prende in input un RDD, per ogni elemento del RDD applica la funzione e crea un altro RDD in output. 

Si vuole creare un applicazione che tramite un modello di ML predice la durata delle corse 

In [ ]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green.select(columns).rdd

In [ ]:
def appply_model_in_batch(partition): 
    #NOTA: partition è un iteratore che contiene un numero di righe del dataframe (Row)
    cont = 0 
    for i in partition:
        cont += 1
    return [cont] 

In [ ]:
rdd.mapPartitions(appply_model_in_batch).collect()

Adesso si vuole trasformare questo RDD in un Dataframe Pandas

In [ ]:
import pandas as pd 

In [ ]:
rows = duration_rdd.take(10)

In [ ]:
rows

In [ ]:
pd.DataFrame(rows, columns=columns)

In [ ]:
def appply_model_in_batch(rows): 
    df = pd.DataFrame(rows, columns=columns)
    cont = len(df)
    return [cont] 

In [ ]:
duration_rdd.mapPartitions(appply_model_in_batch).collect()

Si definisce il Model per fare la previsione tramite ML

In [ ]:
#model = ...
def model_predict(df): 
    #y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [ ]:
def appply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['models_predictions'] = predictions

    for row in df.itertuples():
        yield row

Yield:

In Python, yield trasforma una funzione in un generatore, permettendo di restituire valori uno alla volta, sospendendo l'esecuzione e mantenendo lo stato locale. A differenza di return, yield non termina la funzione, ma la mette in pausa, ottimizzando la memoria per gestire grandi flussi di dati senza caricarli tutti insieme.

Caratteristiche principali di yield:

    - Esecuzione "Pigra" (Lazy Evaluation): I valori vengono generati su richiesta (on-demand), non pre-calcolati.

    - Risparmio di Memoria: Ideale per iterare su liste enormi o file di grandi dimensioni.
    
    - Stato Conservato: Quando la funzione viene ripresa (ad esempio in un ciclo for), riparte esattamente da dove si era interrotta.

Differenze chiave con return:

    - return: Termina definitivamente l'esecuzione della funzione e restituisce un valore.
    - yield: Sospende la funzione, restituendo un valore, e ne permette la ripresa.

In [ ]:
df_predicts = duration_rdd.mapPartitions(appply_model_in_batch).toDF().drop('Index')

In [ ]:
#mostrare i valori di predizione della durata del viaggio
df_predicts.select('models_predictions').show()